In [ ]:
import tensorflow as tf

print("TF version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

if gpus:
    for i, g in enumerate(gpus):
        details = tf.config.experimental.get_device_details(g)
        print(f"GPU {i} details:", details)
else:
    print("Nenhuma GPU visível para o TensorFlow.")

# Abordagem 1: Detecção de outliers (Clustering)

In [ ]:
# Experimento E4: One-Class SVM (tabular) — esqueleto
# Ideia: treinar só na classe "normal" (ex.: sMCI) e usar o score como anomalia.
# Avaliação: AUC para detectar pMCI como "anomalia" (pMCI=1).

from sklearn.svm import OneClassSVM

NORMAL_LABEL = "sMCI"  # escolha: "sMCI" como normal e "pMCI" como anomalia
nu = 0.1
kernel = "rbf"
gamma = "scale"

rows = []
for fold in range(N_SPLITS):
    tr, va = load_fold(fold, selected=True)  # geralmente melhor após seleção
    X_tr, y_tr = split_xy(tr)
    X_va, y_va = split_xy(va)

    # garante mesmas colunas
    common = [c for c in X_tr.columns if c in X_va.columns]
    X_tr = X_tr[common]
    X_va = X_va[common]

    # treina apenas no "normal" (0 = sMCI no split_xy)
    normal_code = 0 if NORMAL_LABEL == "sMCI" else 1
    X_tr_norm = X_tr[y_tr == normal_code]

    if X_tr_norm.shape[0] < 10:
        raise SystemExit(f"Fold {fold}: poucas amostras normais para OneClassSVM: {X_tr_norm.shape[0]}")

    oc = OneClassSVM(nu=nu, kernel=kernel, gamma=gamma)
    oc.fit(X_tr_norm)

    # score_samples: maior = mais inlier; para anomalia usamos o negativo
    inlier_score = oc.score_samples(X_va)
    anomaly_score = -inlier_score

    # y_va já é 0/1 (0=sMCI,1=pMCI). Se normal for pMCI, inverta o alvo.
    y_anom = y_va if NORMAL_LABEL == "sMCI" else (1 - y_va)

    auc = float(roc_auc_score(y_anom, anomaly_score))
    rows.append(
        {
            "fold": fold,
            "model": "OneClassSVM",
            "normal": NORMAL_LABEL,
            "auc_anomaly": auc,
            "n_features": int(X_tr.shape[1]),
            "n_train_normal": int(X_tr_norm.shape[0]),
            "nu": float(nu),
            "kernel": kernel,
        }
    )

pd.DataFrame(rows)


In [ ]:
# One-Class Graph SVM — esqueleto (Graph Kernel + OneClassSVM)
# Estratégia padrão: representar cada amostra como um grafo e usar um kernel de grafos.
# Depois, treinar OneClassSVM com kernel='precomputed' no conjunto de grafos "normais".
#
# OBS: isso normalmente usa a lib GraKeL. Se não estiver instalada, o bloco não roda,
# mas o esqueleto fica pronto.

NORMAL_LABEL = "sMCI"  # classe considerada "normal"

try:
    import networkx as nx
    from grakel import Graph
    from grakel.kernels import WeisfeilerLehman, VertexHistogram
except Exception as e:
    raise SystemExit(
        "Dependências para Graph SVM ausentes. Instale 'grakel' e 'networkx' para rodar este experimento."
    ) from e

from sklearn.svm import OneClassSVM


def build_nx_graph_from_sample(df_sample: pd.DataFrame) -> nx.Graph:
    """Converte uma amostra (linhas ROI/pair) em um grafo.

    Este é um ESQUELETO: você precisa definir:
    - quais são os nós (ex.: 20 ROIs x side)
    - como ligar as arestas (ex.: fully-connected, atlas, kNN por distância)
    - quais features vão para cada nó

    Sugestão rápida: agregar por (roi,side) e usar a média das colunas numéricas como atributo do nó.
    """
    g = nx.Graph()

    # 1) agrega por nó
    keys = [k for k in ["roi", "side"] if k in df_sample.columns]
    if not keys:
        raise ValueError("A amostra não tem colunas 'roi'/'side' para definir nós.")

    X = df_sample.drop(columns=[c for c in NON_FEATURE_COLS if c in df_sample.columns], errors="ignore")
    X = X.apply(pd.to_numeric, errors="coerce")

    agg = pd.concat([df_sample[keys], X], axis=1).groupby(keys, as_index=False).mean(numeric_only=True)

    # 2) adiciona nós com atributos
    node_ids = []
    for _, row in agg.iterrows():
        node = f"{row['roi']}_{row['side']}" if 'side' in agg.columns else str(row['roi'])
        node_ids.append(node)
        attrs = row.drop(labels=keys).to_dict()
        # GraKeL usa rótulos/atributos; aqui colocamos atributos numéricos e também um label fixo.
        g.add_node(node, attr=attrs)

    # 3) arestas (esqueleto: fully-connected)
    for i in range(len(node_ids)):
        for j in range(i + 1, len(node_ids)):
            g.add_edge(node_ids[i], node_ids[j])

    return g


def nx_to_grakel(g: nx.Graph) -> Graph:
    # GraKeL Graph: (adjacency, node_labels)
    # Para simplificar o esqueleto, usamos um label constante por nó.
    nodes = list(g.nodes())
    idx = {n: i for i, n in enumerate(nodes)}

    adj = {idx[u]: [idx[v] for v in g.neighbors(u)] for u in nodes}
    labels = {idx[u]: 1 for u in nodes}  # rótulos discretos (necessário p/ WL+VH)
    return Graph(adj, node_labels=labels)


# Kernel: Weisfeiler-Lehman + VertexHistogram (bom baseline)
KERNEL = WeisfeilerLehman(n_iter=3, base_graph_kernel=VertexHistogram, normalize=True)

rows = []
for fold in range(N_SPLITS):
    tr, va = load_fold(fold, selected=False)

    # 1) transforma cada conjunto (ID_PT,COMBINATION_NUMBER,TRIPLET_IDX) em um grafo
    set_keys = ["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX"]
    tr_sets = tr[set_keys + ["GROUP"]].drop_duplicates(subset=set_keys)
    va_sets = va[set_keys + ["GROUP"]].drop_duplicates(subset=set_keys)

    def build_graphs(df_all: pd.DataFrame, df_sets: pd.DataFrame):
        Gs = []
        ys = []
        for _, r in df_sets.iterrows():
            mask = (df_all["ID_PT"] == r["ID_PT"]) & (df_all["COMBINATION_NUMBER"] == r["COMBINATION_NUMBER"]) & (df_all["TRIPLET_IDX"] == r["TRIPLET_IDX"])
            sample = df_all.loc[mask]
            g_nx = build_nx_graph_from_sample(sample)
            Gs.append(nx_to_grakel(g_nx))
            ys.append(str(r["GROUP"]).strip())
        return Gs, np.array(ys)

    G_tr, y_tr_s = build_graphs(tr, tr_sets)
    G_va, y_va_s = build_graphs(va, va_sets)

    # 2) kernel matrices
    K_tr = KERNEL.fit_transform(G_tr)          # (n_tr, n_tr)
    K_va = KERNEL.transform(G_va)              # (n_va, n_tr)

    # 3) one-class: treina só nos normais
    y_tr_norm = (y_tr_s == NORMAL_LABEL)
    if y_tr_norm.sum() < 5:
        raise SystemExit(f"Fold {fold}: poucas amostras normais para OneClass Graph SVM: {int(y_tr_norm.sum())}")

    oc = OneClassSVM(kernel="precomputed", nu=0.1)
    oc.fit(K_tr[np.ix_(y_tr_norm, y_tr_norm)])

    # 4) score no val: precisamos do kernel (val x train_normal)
    K_va_norm = K_va[:, y_tr_norm]
    inlier_score = oc.score_samples(K_va_norm)
    anomaly_score = -inlier_score

    y_va_anom = (y_va_s != NORMAL_LABEL).astype(int)
    auc = float(roc_auc_score(y_va_anom, anomaly_score))

    rows.append({"fold": fold, "model": "OneClassGraphSVM(WL)", "auc_anomaly": auc, "normal": NORMAL_LABEL, "n_train_graphs": len(G_tr), "n_val_graphs": len(G_va)})

pd.DataFrame(rows)


# Abordagem 4: Classificação binária sMCI vs pMCI

## Modelagem Tabular

### Pipeline tabular (deltas) — preparação para sMCI vs pMCI

Este bloco carrega `all_delta_features_neurocombat.csv`, codifica os rótulos, reduz redundância (variância e correlação) e ilustra **z-score** sem *data leakage* num **primeiro fold** de `StratifiedGroupKFold` agrupado por `ID_PT`.

**Nota metodológica:** em validação cruzada real, cada etapa que *aprende* do dado (`VarianceThreshold`, remoção por correlação, `StandardScaler`, seleção supervisionada) deve ser **ajustada apenas no treino** de cada fold e **aplicada** (transform / mesma máscara de colunas) na validação e no teste. Aqui, variância e correlação são demonstradas no **treino do fold 0**; o conjunto completo serve só para carregamento e para gráficos de codificação de `GROUP`/`SEX`.


In [4]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

ab = "abordagem_teste"

csv_deltas = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_delta_features_neurocombat_wide.csv"

# CSV_DELTA = ROOT / "csvs" / "abordagem_teste" / "all_delta_features_neurocombat.csv"
# RNG_SEED = 42

pd.set_option("display.max_columns", 20)

df_raw = pd.read_csv(csv_deltas)
# print("CSV:", csv_deltas.resolve())
print("shape:", df_raw.shape)
df_raw.head(2)


shape: (1840, 463)


,ID_PT,COMBINATION_NUMBER,TRIPLET_IDX,roi,side,GROUP,SEX,centroid_x__d12,centroid_x__d13,centroid_x__d23,...,uz_p05__d23,uz_p50__d12,uz_p50__d13,uz_p50__d23,uz_p95__d12,uz_p95__d13,uz_p95__d23,uz_std__d12,uz_std__d13,uz_std__d23
0,002_S_0729,1,0,accumbens_area,L,pMCI,F,9.717575,9.722203,15.696340,...,1.252484,1.943424,3.223910,1.519447,2.367145,3.607919,1.870913,0.280007,0.244869,0.183868
1,002_S_0729,1,0,accumbens_area,R,pMCI,F,-7.204837,-7.200209,-1.318937,...,1.062809,1.630100,2.947738,1.253053,2.210951,3.251310,1.534164,0.299490,0.203840,0.129704


#### Codificação de `GROUP` e `SEX`

- `GROUP`: **sMCI = 0**, **pMCI = 1**
- `SEX`: **F = 0**, **M = 1**

Linhas com valores fora do esperado são removidas e contabilizadas. Os gráficos comparam frequências **antes** (strings) e **depois** (inteiro).


In [ ]:
GROUP_MAP = {"sMCI": 0, "pMCI": 1}
SEX_MAP = {"F": 0, "M": 1}

_df = df_raw.copy()
_df["GROUP"] = _df["GROUP"].astype(str).str.strip()
_df["SEX"] = _df["SEX"].astype(str).str.strip()

mask_group = _df["GROUP"].isin(GROUP_MAP)
mask_sex = _df["SEX"].isin(SEX_MAP)
valid = mask_group & mask_sex
n_drop = int((~valid).sum())
if n_drop:
    print("Linhas removidas (GROUP/SEX inválidos):", n_drop)
    print("GROUP inválidos:", sorted(_df.loc[~mask_group, "GROUP"].unique().tolist())[:20])
    print("SEX inválidos:", sorted(_df.loc[~mask_sex, "SEX"].unique().tolist())[:20])

df = _df.loc[valid].reset_index(drop=True)
df["y_group"] = df["GROUP"].map(GROUP_MAP).astype(np.int8)
df["sex_enc"] = df["SEX"].map(SEX_MAP).astype(np.int8)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
vc_g = df["GROUP"].value_counts().reindex(["sMCI", "pMCI"]).fillna(0)
ax.bar(["sMCI", "pMCI"], vc_g.values, color=["#3498db", "#e74c3c"])
ax.set_title("GROUP (antes — strings)")
ax.set_ylabel("N linhas")

ax = axes[1]
vc_y = df["y_group"].value_counts().sort_index()
ax.bar(["0 sMCI", "1 pMCI"], vc_y.reindex([0, 1]).fillna(0).values, color=["#3498db", "#e74c3c"])
ax.set_title("y_group (depois)")
ax.set_ylabel("N linhas")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax = axes[0]
vc_s = df["SEX"].value_counts().reindex(["F", "M"]).fillna(0)
ax.bar(["F", "M"], vc_s.values, color=["#9b59b6", "#34495e"])
ax.set_title("SEX (antes — strings)")
ax.set_ylabel("N linhas")

ax = axes[1]
vc_se = df["sex_enc"].value_counts().sort_index()
ax.bar(["0 F", "1 M"], vc_se.reindex([0, 1]).fillna(0).values, color=["#9b59b6", "#34495e"])
ax.set_title("sex_enc (depois)")
ax.set_ylabel("N linhas")
plt.tight_layout()
plt.show()

ct = pd.crosstab(df["y_group"], df["sex_enc"], rownames=["y_group"], colnames=["sex_enc"])
print(ct)
try:
    from IPython.display import display

    display(ct)
except Exception:
    pass


#### Matriz de atributos numéricos (radiomorfometria + covariável de sexo)

Colunas de metadados / identificação são excluídas de **X**. O alvo binário fica em `y_group`; `sex_enc` permanece em **X** como covariável (0/1).

Valores não numéricos nas colunas de atributos são coagidos com `to_numeric`; NaN resultantes são preenchidos com **0.0** (substituível por imputação por mediana *só no treino* num pipeline formal).


In [ ]:
NON_FEATURE_COLS = [
    "ID_PT",
    "COMBINATION_NUMBER",
    "TRIPLET_IDX",
    "pair",
    "ID_IMG_i1",
    "ID_IMG_i2",
    "ID_IMG_i3",
    "roi",
    "side",
    "label",
    "t12", # usar para ponderar os atributos
    "t13", # usar para ponderar os atributos
    "t23", # usar para ponderar os atributos
    "GROUP",
    "SEX",
    "DIAG",
    "AGE",
    "TIME_PROG",
    "ID_IMG_ref",
    "FIELD_STRENGTH",
    "SLICE_THICKNESS",
    "MANUFACTURER",
    "MFG_MODEL",
    "batch",
    "centroid_x", # atributos de posição
    "centroid_y", # atributos de posição
    "centroid_z", # atributos de posição
    "y_group",
]

missing_non_feature = [c for c in NON_FEATURE_COLS if c not in df.columns]
if missing_non_feature:
    warnings.warn(f"Colunas em NON_FEATURE_COLS ausentes no CSV: {missing_non_feature}")

feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
X_all = df[feature_cols].apply(pd.to_numeric, errors="coerce")
n_nan_cell = int(X_all.isna().sum().sum())
if n_nan_cell:
    print(f"Total de NaN em X (após to_numeric): {n_nan_cell} — imputando 0.0 para demo")
X_all = X_all.fillna(0.0)
y_all = df["y_group"].to_numpy(dtype=np.int64)
groups_all = df["ID_PT"].to_numpy()

N_FEATURES_RAW = X_all.shape[1]
print("N atributos numéricos (X):", N_FEATURES_RAW)

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Atributos\n(inicial)"], [N_FEATURES_RAW], color="#3498db")
ax.set_ylabel("Contagem")
ax.set_title("Número de colunas em X antes de variância/correlação")
for i, v in enumerate([N_FEATURES_RAW]):
    ax.text(i, v + N_FEATURES_RAW * 0.01, str(v), ha="center")
plt.tight_layout()
plt.show()


#### Remoção de baixa variância (`VarianceThreshold`)

**Demonstração no treino do fold 0** (`StratifiedGroupKFold`, `groups=ID_PT`): o limiar remove colunas com variância inferior ao valor escolhido (por defeito `0.0` remove apenas constantes; use `1e-10` ou `1e-8` para quase-constantes).

Histograma: variâncias amostrais **por coluna** apenas no **treino** do fold 0 (antes do filtro).


In [ ]:
# Split por paciente com estratificação pelo alvo binário (y_group).
# Para estratificar também por sexo, use um y composto (LabelEncoder) se cada strato tiver amostras suficientes por fold.
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RNG_SEED)
train_idx, val_idx = next(sgkf.split(X_all, y_all, groups=groups_all))
X_tr = X_all.iloc[train_idx].copy()
X_va = X_all.iloc[val_idx].copy()
y_tr = y_all[train_idx]
y_va = y_all[val_idx]
print("Fold 0 — treino:", X_tr.shape[0], "validação:", X_va.shape[0], "| pacientes únicos tr:", df.iloc[train_idx]["ID_PT"].nunique())

VARIANCE_THRESHOLD = 0.0  # ajuste p.ex. para 1e-10 se quiser cortar quase-constantes
col_variances = X_tr.var(axis=0, numeric_only=True).to_numpy()

vt = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
vt.fit(X_tr)
support = vt.get_support()
cols_after_vt = X_tr.columns[support].tolist()
n_after_vt = len(cols_after_vt)
print("Após VarianceThreshold:", n_after_vt, "de", X_tr.shape[1])

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(np.log10(col_variances + 1e-20), bins=50, color="#95a5a6", edgecolor="white")
ax.axvline(np.log10(VARIANCE_THRESHOLD + 1e-20), color="red", ls="--", label=f"log10(limiar+eps)={VARIANCE_THRESHOLD}")
ax.set_xlabel("log10(variância por coluna no treino)")
ax.set_ylabel("N colunas")
ax.set_title("Distribuição de variâncias (treino fold 0, antes do filtro)")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
labels = ["Antes\n(treino fold0)", "Depois\nVarianceThreshold"]
vals = [X_tr.shape[1], n_after_vt]
ax.bar(labels, vals, color=["#3498db", "#9b59b6"])
ax.set_ylabel("N atributos")
ax.set_title("Baixa variância — contagem de colunas (treino)")
for i, v in enumerate(vals):
    ax.text(i, v + max(vals) * 0.02, str(v), ha="center")
plt.tight_layout()
plt.show()

X_tr_vt = X_tr[cols_after_vt]
X_va_vt = X_va[cols_after_vt]


#### Remoção de alta correlação (atributo × atributo)

No **mesmo treino do fold 0**, após variância: para cada par de colunas com **|correlação de Pearson| > 0,95**, remove-se uma delas (a segunda na ordem lexical) para reduzir redundância. O mesmo conjunto de colunas retido é aplicado à validação.

*Heatmap*: amostra de até 35 colunas após o filtro (se houver mais, ordem lexical).


In [ ]:
CORR_THRESHOLD = 0.90


def drop_high_correlation(X: pd.DataFrame, threshold: float):
    # Remove uma coluna de cada par com |corr| > threshold (greedy por ordem de colunas).
    Xn = X.copy()
    cols = Xn.columns.tolist()
    corr = Xn.corr(method="pearson").abs()
    to_drop = set()
    for i, ci in enumerate(cols):
        if ci in to_drop:
            continue
        for j in range(i + 1, len(cols)):
            cj = cols[j]
            if cj in to_drop:
                continue
            if pd.isna(corr.loc[ci, cj]):
                continue
            if corr.loc[ci, cj] > threshold:
                to_drop.add(cj)
    keep = [c for c in cols if c not in to_drop]
    return Xn[keep], sorted(to_drop)


n_before_corr = X_tr_vt.shape[1]
X_tr_corr, dropped_corr = drop_high_correlation(X_tr_vt, CORR_THRESHOLD)
cols_after_corr = X_tr_corr.columns.tolist()
X_va_corr = X_va_vt[cols_after_corr]
n_after_corr = len(cols_after_corr)
print(f"Colunas removidas por correlação > {CORR_THRESHOLD}:", len(dropped_corr))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Antes\n(treino)", "Depois\ncorr>" + str(CORR_THRESHOLD)], [n_before_corr, n_after_corr], color=["#9b59b6", "#e67e22"])
ax.set_ylabel("N atributos")
ax.set_title("Alta correlação — contagem de colunas (treino fold 0)")
for i, v in enumerate([n_before_corr, n_after_corr]):
    ax.text(i, v + max(n_before_corr, n_after_corr) * 0.02, str(v), ha="center")
plt.tight_layout()
plt.show()

try:
    import seaborn as sns

    k = min(35, X_tr_corr.shape[1])
    sub = X_tr_corr.iloc[:, :k]
    cm = sub.corr()
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, ax=ax, cmap="vlag", center=0, xticklabels=False, yticklabels=False, cbar_kws={"shrink": 0.6})
    ax.set_title(f"Heatmap |corr| (primeiras {k} colunas após filtro)")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("seaborn não instalado — heatmap omitido.")


#### Z-score (`StandardScaler`) sem *leakage*

`fit` apenas em **treino** do fold 0 (já após variância + correlação); `transform` em treino e validação. Os gráficos mostram a distribuição de **três colunas** escolhidas por maior variância no treino (antes vs depois do scale), apenas no **treino**.


In [ ]:
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_corr)
X_va_scaled = scaler.transform(X_va_corr)
X_tr_scaled = pd.DataFrame(X_tr_scaled, columns=cols_after_corr, index=X_tr_corr.index)
X_va_scaled = pd.DataFrame(X_va_scaled, columns=cols_after_corr, index=X_va_corr.index)

# Três colunas com maior variância no treino (antes do scale)
top3 = X_tr_corr.var().sort_values(ascending=False).head(3).index.tolist()
fig, axes = plt.subplots(len(top3), 2, figsize=(10, 3 * len(top3)))
if len(top3) == 1:
    axes = np.array([axes])
for r, col in enumerate(top3):
    axes[r, 0].hist(X_tr_corr[col], bins=40, color="#3498db", alpha=0.85)
    axes[r, 0].set_title(f"{col[:40]}…\n(treino, antes)" if len(col) > 40 else f"{col}\n(treino, antes)")
    axes[r, 1].hist(X_tr_scaled[col], bins=40, color="#2ecc71", alpha=0.85)
    axes[r, 1].set_title("treino, depois z-score")
plt.tight_layout()
plt.show()

print("Média (aprox.) no treino após scale:", np.mean(X_tr_scaled.values, axis=0)[:5], "...")
print("DP (aprox.) no treino após scale:", np.std(X_tr_scaled.values, axis=0, ddof=0)[:5], "...")


#### Resumo da redução de atributos e exportação opcional

Contagens referem-se ao **treino do fold 0** após a matriz **X** inicial (número de colunas iguais ao global). Opcionalmente grava-se o treino do fold 0 com atributos finais + metadados mínimos (sem *leakage* de scaler para outras divisões).


In [ ]:
summary_counts = {
    "1_X_inicial": N_FEATURES_RAW,
    "2_após_variância": n_after_vt,
    "3_após_correlação": n_after_corr,
}

fig, ax = plt.subplots(figsize=(8, 4))
steps = list(summary_counts.keys())
vals = list(summary_counts.values())
ax.bar(steps, vals, color=["#3498db", "#9b59b6", "#e74c3c"])
ax.set_ylabel("N atributos")
ax.set_title("Redução de atributos (treino fold 0, após passos 2–3)")
ax.tick_params(axis="x", rotation=15)
for i, v in enumerate(vals):
    ax.text(i, v + max(vals) * 0.02, str(v), ha="center")
plt.tight_layout()
plt.show()

# Export opcional: apenas treino fold 0, alinhado ao scaler/ colunas acima
EXPORT = False  # mude para True para gravar CSV
if EXPORT:
    out = df.iloc[train_idx][["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX", "roi", "side", "GROUP", "SEX", "y_group", "sex_enc"]].copy()
    out = pd.concat([out.reset_index(drop=True), X_tr_scaled.reset_index(drop=True)], axis=1)
    out_path = ROOT / "csvs" / "abordagem_teste" / "all_delta_tabular_fold0_train_scaled.csv"
    out.to_csv(out_path, index=False)
    print("Gravado:", out_path.resolve())


#### Relação com os experimentos E1–E3 (abaixo)

As células **Elastic Net**, **SVM linear** e **XGBoost** chamam `load_fold`, `split_xy` e `N_SPLITS`, que **não estão definidos** neste notebook. O próximo passo metodológico é encapsular o fluxo acima num `sklearn.pipeline.Pipeline` (por exemplo `VarianceThreshold` + transformador custom para correlação + `StandardScaler` + classificador) e executá-lo **dentro** de cada iteração de `StratifiedGroupKFold`, com `fit` apenas no treino de cada fold — reproduzindo a mesma lógica demonstrada aqui no fold 0.


## Modelagem longitudinal

In [ ]:
# Longitudinal 2 (deltas 12/13/23) a partir do parquet wide
# Monta X_seq com shape (N, T=3, F_step) onde T=[12,13,23]

# OBS: requer pandas com suporte a parquet (pyarrow/fastparquet).

PAIR_ORDER = ["12", "13", "23"]

if not PARQUET_WIDE.is_file():
    raise SystemExit(f"Parquet não encontrado: {PARQUET_WIDE}")

wide = pd.read_parquet(PARQUET_WIDE)

# IDs e label
id_cols = [c for c in ["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX", "GROUP"] if c in wide.columns]

# identifica colunas por pair via sufixo '__..._p12'
feat_cols = [c for c in wide.columns if "__" in c]

pair_to_cols = {}
for p in PAIR_ORDER:
    suf = f"_p{p}"
    cols = [c for c in feat_cols if c.endswith(suf)]
    pair_to_cols[p] = cols

# garante mesmo conjunto de 'features base' entre pares (pode haver pequenas diferenças)
min_cols = min(len(v) for v in pair_to_cols.values())
print({p: len(cols) for p, cols in pair_to_cols.items()}, "min=", min_cols)

# para comparabilidade: mantém a interseção de colunas dos 3 pares
common_cols = set(pair_to_cols[PAIR_ORDER[0]])
for p in PAIR_ORDER[1:]:
    common_cols &= set(pair_to_cols[p])
common_cols = sorted(common_cols)

# reordena para cada p, mantendo mesma base de features
pair_cols_ordered = {p: [c for c in common_cols if c.endswith(f"_p{p}")] for p in PAIR_ORDER}

print("common per pair", {p: len(v) for p, v in pair_cols_ordered.items()})

# Monta tensor (aqui como numpy) — você vai alimentar no PyTorch depois
X_steps = []
for p in PAIR_ORDER:
    Xp = wide[pair_cols_ordered[p]].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    X_steps.append(Xp)

X_seq = np.stack(X_steps, axis=1)  # (N, 3, F)

y = wide["GROUP"].astype(str).map({"sMCI": 0, "pMCI": 1}).to_numpy(dtype=np.int64)

print("X_seq", X_seq.shape, "y", y.shape)


In [ ]:
# Longitudinal 1 (t1,t2,t3): monta sequência por imagem usando features_merged_combat.csv
# Produz um vetor por tempo agregando por imagem (média sobre ROIs) — baseline rápido.

FEATURES_IMG = BASE / "csvs/features_merged_combat.csv"

if not FEATURES_IMG.is_file():
    raise SystemExit(f"Arquivo ausente: {FEATURES_IMG}")

feat_img = pd.read_csv(FEATURES_IMG)
need = {"ID_IMG", "roi", "side", "label"}
if not need.issubset(set(feat_img.columns)):
    raise SystemExit(f"features_merged_combat.csv sem colunas {sorted(need - set(feat_img.columns))}")

# pega apenas colunas numéricas
meta = {"ID_IMG", "roi", "side", "label"}
num_cols = []
Z = feat_img.copy()
for c in Z.columns:
    if c in meta:
        continue
    s = Z[c]
    if pd.api.types.is_numeric_dtype(s):
        num_cols.append(c)
    else:
        s_num = pd.to_numeric(s, errors="coerce")
        if s_num.notna().any():
            Z[c] = s_num
            num_cols.append(c)

# agrega por imagem (média sobre ROIs)
img_vec = Z.groupby("ID_IMG", as_index=True)[num_cols].mean()
print("n_images", img_vec.shape[0], "F", img_vec.shape[1])

# Exemplo: montar sequência para um fold (repita por fold para CV-safe)
fold = 0
tr, va = load_fold(fold, selected=False)

# conjuntos únicos no fold
keys = ["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX", "GROUP", "ID_IMG_i1", "ID_IMG_i2", "ID_IMG_i3"]
tr_sets = tr[keys].drop_duplicates(subset=["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX"]).copy()
va_sets = va[keys].drop_duplicates(subset=["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX"]).copy()


def build_seq(df_sets: pd.DataFrame):
    ids1 = df_sets["ID_IMG_i1"].astype(str)
    ids2 = df_sets["ID_IMG_i2"].astype(str)
    ids3 = df_sets["ID_IMG_i3"].astype(str)

    X1 = img_vec.reindex(ids1).to_numpy(dtype=np.float32)
    X2 = img_vec.reindex(ids2).to_numpy(dtype=np.float32)
    X3 = img_vec.reindex(ids3).to_numpy(dtype=np.float32)

    X = np.stack([X1, X2, X3], axis=1)  # (N,3,F)
    y = df_sets["GROUP"].astype(str).map({"sMCI": 0, "pMCI": 1}).to_numpy(dtype=np.int64)
    return X, y

X_tr_seq, y_tr = build_seq(tr_sets)
X_va_seq, y_va = build_seq(va_sets)

print("train", X_tr_seq.shape, "val", X_va_seq.shape)
print("missing rows train", np.isnan(X_tr_seq).any(axis=(1,2)).sum(), "val", np.isnan(X_va_seq).any(axis=(1,2)).sum())


## Modelagem Grafos

In [ ]:
# Grafos inter-tempo (esqueleto): construir dados para PyTorch Geometric
# Nós: roi_side (20)
# Features por nó: exemplo usando deltas do parquet wide (p12/p13/p23)

# Este bloco só prepara tensores; o treinamento GNN (GCN/GraphSAGE/GAT) vem depois.

# 1) Define a ordem de nós (roi_side) a partir do features_all do seu parquet (recomendado fixar via ROI_TABLE)
# Aqui: extrai dos nomes de colunas do parquet wide.

feat_cols = [c for c in wide.columns if "__" in c]

# recupera sufixos únicos "roi_side_pair" dos nomes das colunas
suffixes = sorted({c.split("__",1)[1] for c in feat_cols})
# reduz a roi_side (removendo _p12/_p13/_p23)
roi_side = sorted({s.rsplit("_p",1)[0] for s in suffixes})
print("n_roi_side", len(roi_side))

# 2) para cada roi_side, pegue features de cada pair e concatene no nó
# Ex.: node_feat = [all_features_p12, all_features_p13, all_features_p23]

# identifica a base de feature (prefixo antes de '__')
base_feats = sorted({c.split("__",1)[0] for c in feat_cols})

# para manter tamanho manejável, você provavelmente vai selecionar um subconjunto (via seleção por fold)
print("base_feats", len(base_feats))

# exemplo: construir matriz de nós para 1 amostra (índice 0)
idx = 0
node_features = []
for rs in roi_side:
    vec_parts = []
    for p in PAIR_ORDER:
        cols = [f"{bf}__{rs}_p{p}" for bf in base_feats if f"{bf}__{rs}_p{p}" in wide.columns]
        x = wide.loc[idx, cols].to_numpy(dtype=np.float32)
        vec_parts.append(x)
    node_features.append(np.concatenate(vec_parts, axis=0))

X_nodes = np.stack(node_features, axis=0)  # (n_nodes, F_node)
print("X_nodes", X_nodes.shape)

# 3) arestas (edge_index) — comece com um grafo fixo simples (ex.: fully-connected sem self-loop) ou um atlas
# Ex.: fully connected (custo O(N^2), mas N=20 é pequeno)

edges = []
for i in range(len(roi_side)):
    for j in range(len(roi_side)):
        if i == j:
            continue
        edges.append([i, j])
edge_index = np.array(edges, dtype=np.int64).T  # (2, E)
print("edge_index", edge_index.shape)

# 4) y
print("y", int(wide.loc[idx, "GROUP"]) if "GROUP" in wide.columns else None)
